## Prepare Env

In [17]:
!python3 -m venv venv
!source venv/bin/activate

In [18]:
%pip install boto3 pyspark delta-spark python-dotenv

  Obtaining dependency information for boto3 from https://files.pythonhosted.org/packages/e4/ad/0f51ad6931032eaca0076000c64351752e222140255eae633f0e037aec45/boto3-1.34.20-py3-none-any.whl.metadata
  Using cached boto3-1.34.20-py3-none-any.whl.metadata (6.6 kB)
  Using cached pyspark-3.5.0-py2.py3-none-any.whl
  Obtaining dependency information for delta-spark from https://files.pythonhosted.org/packages/b2/1b/7b7e4fafc7af2c04fee0c3b51a148104365d2f2eb3aff0bc22a7ccc068be/delta_spark-3.0.0-py3-none-any.whl.metadata
  Using cached delta_spark-3.0.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached python_dotenv-1.0.0-py3-none-any.whl (19 kB)
Using cached boto3-1.34.20-py3-none-any.whl (139 kB)
Using cached delta_spark-3.0.0-py3-none-any.whl (21 kB)

[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
from dotenv import load_dotenv

In [20]:
load_dotenv()

False

In [22]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://localhost:9000')

## Ingestion
### 1. Layer files to layer bronze
Write files which are in layer files to delta table in layer bronze

In [23]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("CsvToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [26]:
file_path = "s3a://warehouse/files/gleif.file/data.csv"
delta_table_path = "s3a://warehouse/bronze/gleif_entities.delta"

In [29]:
# Read file into a DataFrame
df = spark.read.csv(file_path, header=True, sep=",")

In [30]:
df.show()

+--------------------+---------+----+-----+---------+------+--------------------+------------------------------+-------------------------+--------------------------+----------------------+-----------+--------------------+-----------------------+
|        Company Name|      ACN|Type|Class|Sub Class|Status|Date of Registration|Previous State of Registration|State Registration number|Modified since last report|Current Name Indicator|        ABN|        Current Name|Current Name Start Date|
+--------------------+---------+----+-----+---------+------+--------------------+------------------------------+-------------------------+--------------------------+----------------------+-----------+--------------------+-----------------------+
|TGR BIOSCIENCES P...|097258789|APTY| LMSH|     PROP|  REGD|          25/06/2001|                          NULL|                 T1377600|                      NULL|                     Y|39097258789|                NULL|                   NULL|
|ILLUMINATED SOL

In [12]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

24/01/17 15:55:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/01/17 15:55:47 WARN MemoryManager: Total allocation exceeds 95.00% 

# Processing

Processing these step before writing data to layer silver
1. Transform to standard schema of layer silver
2. Unique each record
3. Add fields
4. Map entities
5. Upsert

Read delta table and discovery data

In [13]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("CsvToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

24/01/17 15:57:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [14]:
delta_table_path = "s3a://warehouse/bronze/gleif_entities.delta"

In [15]:
# Read Delta table
df = spark.read.format("delta").load(delta_table_path)

In [16]:
df.show()

+--------------------+--------------------+------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------+-------------------------------------------------+----------------------------------------------+---------------------------------------------------------------------+-----------------------------------------------------------------------------+--------------------------------------------------------------------------+--------------